In [1]:
import os
os.chdir(os.getenv("PROJECT_ROOT", "../../.."))

In [2]:
import pandas as pd
import numpy as np
import ast
from langchain_community.vectorstores import FAISS
from langchain_community.docstore import InMemoryDocstore
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.faiss import dependable_faiss_import
import os

os.environ["OPENAI_API_KEY"] = ""

# 1️⃣ CSV 불러오기 및 정제
df = pd.read_csv("logic/results/embeddings.csv")
df["embedding"] = df["embedding"].apply(ast.literal_eval)


In [3]:
from pprint import pprint
pprint(df)

             text     학수번호   분반  \
0     프로그래밍과 문제해결  GED6011  1.0   
1     배터리속 소재 이야기  GED6010  1.0   
2         미디어 글쓰기  COM2241  1.0   
3         뉴스취재보도론  COM2242  1.0   
4          전자회로 1  EEC2108  6.0   
...           ...      ...  ...   
2156       화공기기분석  CHE3305  2.0   
2157        반도체개론  CHE3308  1.0   
2158        병렬컴퓨팅  AIE3013  1.0   
2159     컴퓨터 네트워크  AIE3009  1.0   
2160        금융시장론  BNF2201  1.0   

                                              embedding  
0     [-0.024946624413132668, 0.040936291217803955, ...  
1     [0.003713945159688592, 0.012014275416731834, -...  
2     [0.03798713907599449, 0.03360073268413544, -0....  
3     [0.03333742171525955, 0.05497995764017105, 0.0...  
4     [-0.005150322802364826, 0.02363019995391369, -...  
...                                                 ...  
2156  [0.013276484794914722, 0.07008206099271774, -0...  
2157  [-0.0190107561647892, 0.035866353660821915, -0...  
2158  [-0.011866560205817223, 0.0077860718593001366,...  

In [4]:

# 결측값 제거
df = df[df["학수번호"].notna()].reset_index(drop=True)
pprint(df)


             text     학수번호   분반  \
0     프로그래밍과 문제해결  GED6011  1.0   
1     배터리속 소재 이야기  GED6010  1.0   
2         미디어 글쓰기  COM2241  1.0   
3         뉴스취재보도론  COM2242  1.0   
4          전자회로 1  EEC2108  6.0   
...           ...      ...  ...   
2087       화공기기분석  CHE3305  2.0   
2088        반도체개론  CHE3308  1.0   
2089        병렬컴퓨팅  AIE3013  1.0   
2090     컴퓨터 네트워크  AIE3009  1.0   
2091        금융시장론  BNF2201  1.0   

                                              embedding  
0     [-0.024946624413132668, 0.040936291217803955, ...  
1     [0.003713945159688592, 0.012014275416731834, -...  
2     [0.03798713907599449, 0.03360073268413544, -0....  
3     [0.03333742171525955, 0.05497995764017105, 0.0...  
4     [-0.005150322802364826, 0.02363019995391369, -...  
...                                                 ...  
2087  [0.013276484794914722, 0.07008206099271774, -0...  
2088  [-0.0190107561647892, 0.035866353660821915, -0...  
2089  [-0.011866560205817223, 0.0077860718593001366,...  

In [5]:

# 2️⃣ 분반 문자열 변환
df["분반"] = df["분반"].apply(lambda x: "" if pd.isna(x) else str(int(x)).zfill(3))
pprint(df)

             text     학수번호   분반  \
0     프로그래밍과 문제해결  GED6011  001   
1     배터리속 소재 이야기  GED6010  001   
2         미디어 글쓰기  COM2241  001   
3         뉴스취재보도론  COM2242  001   
4          전자회로 1  EEC2108  006   
...           ...      ...  ...   
2087       화공기기분석  CHE3305  002   
2088        반도체개론  CHE3308  001   
2089        병렬컴퓨팅  AIE3013  001   
2090     컴퓨터 네트워크  AIE3009  001   
2091        금융시장론  BNF2201  001   

                                              embedding  
0     [-0.024946624413132668, 0.040936291217803955, ...  
1     [0.003713945159688592, 0.012014275416731834, -...  
2     [0.03798713907599449, 0.03360073268413544, -0....  
3     [0.03333742171525955, 0.05497995764017105, 0.0...  
4     [-0.005150322802364826, 0.02363019995391369, -...  
...                                                 ...  
2087  [0.013276484794914722, 0.07008206099271774, -0...  
2088  [-0.0190107561647892, 0.035866353660821915, -0...  
2089  [-0.011866560205817223, 0.0077860718593001366,...  

In [6]:

# 3️⃣ 메타데이터 구성 (교과목명 제외)
metadatas = [
    {"학수번호": row["학수번호"], "분반": row["분반"]} for _, row in df.iterrows()
]
pprint(metadatas)

[{'분반': '001', '학수번호': 'GED6011'},
 {'분반': '001', '학수번호': 'GED6010'},
 {'분반': '001', '학수번호': 'COM2241'},
 {'분반': '001', '학수번호': 'COM2242'},
 {'분반': '006', '학수번호': 'EEC2108'},
 {'분반': '001', '학수번호': 'APL2507'},
 {'분반': '001', '학수번호': 'MCT9002'},
 {'분반': '002', '학수번호': 'MCT9006'},
 {'분반': '001', '학수번호': 'MCT9007'},
 {'분반': '002', '학수번호': 'MCT9002'},
 {'분반': '001', '학수번호': 'MCT9006'},
 {'분반': '001', '학수번호': 'PSE4070'},
 {'분반': '001', '학수번호': 'STS4023'},
 {'분반': '001', '학수번호': 'PHE2943'},
 {'분반': '001', '학수번호': 'ENG2403'},
 {'분반': '001', '학수번호': 'ENG2401'},
 {'분반': '002', '학수번호': 'CHN4346'},
 {'분반': '001', '학수번호': 'CHN4346'},
 {'분반': '001', '학수번호': 'CON3606'},
 {'분반': '001', '학수번호': 'IPS3456'},
 {'분반': '001', '학수번호': 'EKR1529'},
 {'분반': '001', '학수번호': 'BUS4102'},
 {'분반': '001', '학수번호': 'SWE3204'},
 {'분반': '001', '학수번호': 'CHM4514'},
 {'분반': '001', '학수번호': 'PHY3032'},
 {'분반': '001', '학수번호': 'STS3050'},
 {'분반': '002', '학수번호': 'CSE2107'},
 {'분반': '001', '학수번호': 'CSE2105'},
 {'분반': '003', '학수번호

In [7]:
print(len(metadatas))

2092


In [10]:
# 4️⃣ 임베딩 및 ID 준비
embeddings = np.vstack(df["embedding"].values).astype("float32")
ids = df["학수번호"].tolist()
pprint(embeddings)
pprint(ids)
print(embeddings.shape)
print(len(ids))

array([[-0.02494662,  0.04093629,  0.03003216, ...,  0.01806282,
        -0.00748513, -0.014913  ],
       [ 0.00371395,  0.01201428, -0.00365722, ...,  0.00376797,
        -0.02651352,  0.00038828],
       [ 0.03798714,  0.03360073, -0.0444177 , ..., -0.01003976,
        -0.00706936,  0.00033703],
       ...,
       [-0.01186656,  0.00778607,  0.02358099, ..., -0.01684356,
        -0.01989713, -0.01418119],
       [-0.04325814,  0.02251647, -0.0156204 , ...,  0.00164383,
        -0.00401202, -0.02219572],
       [-0.02364892,  0.00979741,  0.0044896 , ...,  0.00124711,
         0.00529197, -0.00656152]], dtype=float32)
['GED6011',
 'GED6010',
 'COM2241',
 'COM2242',
 'EEC2108',
 'APL2507',
 'MCT9002',
 'MCT9006',
 'MCT9007',
 'MCT9002',
 'MCT9006',
 'PSE4070',
 'STS4023',
 'PHE2943',
 'ENG2403',
 'ENG2401',
 'CHN4346',
 'CHN4346',
 'CON3606',
 'IPS3456',
 'EKR1529',
 'BUS4102',
 'SWE3204',
 'CHM4514',
 'PHY3032',
 'STS3050',
 'CSE2107',
 'CSE2105',
 'CSE2105',
 'CSE2107',
 'CSE2104',


In [ ]:

# 5️⃣ FAISS 인덱스 생성
faiss = dependable_faiss_import()
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(index.ntotal)

2092


In [12]:
print(len(df))

2092


In [18]:

# 6️⃣ LangChain FAISS wrapping
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
db = FAISS(
    embedding_function=embedding_model,
    index=index,
    docstore=InMemoryDocstore(
        {ids[i]: {"page_content": "", "metadata": metadatas[i]} for i in range(len(df))}
    ),
    index_to_docstore_id={i: ids[i] for i in range(len(df))},
)
print(db.index.ntotal)
print(len(metadatas))
print(db.docstore._dict)
print(help(db.docstore))
docs = db.docstore._dict
pprint(docs)
print(len(docs))

2092
2092
{'GED6011': {'page_content': '', 'metadata': {'학수번호': 'GED6011', '분반': '001'}}, 'GED6010': {'page_content': '', 'metadata': {'학수번호': 'GED6010', '분반': '001'}}, 'COM2241': {'page_content': '', 'metadata': {'학수번호': 'COM2241', '분반': '001'}}, 'COM2242': {'page_content': '', 'metadata': {'학수번호': 'COM2242', '분반': '001'}}, 'EEC2108': {'page_content': '', 'metadata': {'학수번호': 'EEC2108', '분반': '005'}}, 'APL2507': {'page_content': '', 'metadata': {'학수번호': 'APL2507', '분반': '001'}}, 'MCT9002': {'page_content': '', 'metadata': {'학수번호': 'MCT9002', '분반': '002'}}, 'MCT9006': {'page_content': '', 'metadata': {'학수번호': 'MCT9006', '분반': '001'}}, 'MCT9007': {'page_content': '', 'metadata': {'학수번호': 'MCT9007', '분반': '001'}}, 'PSE4070': {'page_content': '', 'metadata': {'학수번호': 'PSE4070', '분반': '001'}}, 'STS4023': {'page_content': '', 'metadata': {'학수번호': 'STS4023', '분반': '001'}}, 'PHE2943': {'page_content': '', 'metadata': {'학수번호': 'PHE2943', '분반': '001'}}, 'ENG2403': {'page_content': '', 'metadata

In [ ]:

# 7️⃣ 저장
db.save_local("faiss_index")
print(f"✅ 새 FAISS 인덱스 저장 완료 (총 {len(df)}개, 교과목명 제외)")
